# Task 2.6 — Nền tảng thí nghiệm DL và baseline tái lập

Chạy trên **Kaggle GPU**, đính kèm competition data và output **task 2.1** để lấy split theo ID. Không đọc test hoặc leaderboard.
Notebook này xuất phần code dùng chung, audit dữ liệu, holdout seed 9, 5 CV folds và baseline chạy lại.
Các notebook 2.7–2.11 phải gắn output của đúng version 2.6 này.

## 2.6.1 — Thư mục output
Đây là file source notebook. Module Python bên dưới là **output tự sinh trên Kaggle**, giúp các task dùng chính xác cùng code; không cần source Python riêng trên laptop.


In [ ]:
from pathlib import Path
if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Open this notebook on Kaggle before running its cells.")
OUTPUT_DIR = Path("/kaggle/working/task_2_6_foundation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2.6.2 — Code dùng chung: cấu hình, kiểm tra input và metric

Cấu hình baseline giữ kiến trúc và tham số task 2.1. Hàm train kiểm tra môi trường Kaggle GPU; import module không tự huấn luyện.
Các hàm hash ngăn dùng nhầm dataset/version hoặc file đã thay đổi. ROC-AUC là metric chính, Average Precision là metric phụ.


In [ ]:
%%writefile /kaggle/working/task_2_6_foundation/credit_fraud_dl_engine.py
"""Shared DL implementation exported by Kaggle task 2.6; no training on import."""
import copy
import gc
import hashlib
import importlib
import json
import platform
import random
import subprocess
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ID, TARGET = "id", "is_fraud"
ENGINE_VERSION = "dl-optimization-v1"

BASELINE = {
    "family": "mlp", "hidden": [128, 64], "dropout": [0.20, 0.10],
    "learning_rate": 0.001, "weight_decay": 0.0001, "batch_size": 4096,
    "max_epochs": 12, "patience": 3, "min_delta": 0.000001,
    "scheduler": False, "scheduler_patience": 2, "scheduler_factor": 0.5,
    "min_learning_rate": 0.000001, "gradient_clip": 1.0,
    "training_seed": 9, "positive_weight": 1.0, "scale_boolean": True,
    "log_columns": [], "cyclical": False, "embedding_dim": None,
    "tabnet": {"n_d": 16, "n_a": 16, "n_steps": 4, "gamma": 1.3,
               "lambda_sparse": 0.0001, "cat_emb_dim": 1, "virtual_batch_size": 512},
}


def json_write(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
                          encoding="utf-8")


def json_read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def value_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def config(**changes):
    result = copy.deepcopy(BASELINE)
    unknown = set(changes) - set(result)
    if unknown:
        raise ValueError(f"Unsupported configuration keys: {unknown}")
    result.update(changes)
    if result["family"] not in {"mlp", "embedding", "tabnet"}:
        raise ValueError("Only DL model families are supported.")
    if len(result["hidden"]) != len(result["dropout"]) or not result["hidden"]:
        raise ValueError("Each hidden layer needs its dropout value.")
    if any(not 0 <= p < 1 for p in result["dropout"]):
        raise ValueError("Dropout must be in [0, 1).")
    if result["learning_rate"] <= 0 or result["batch_size"] < 1:
        raise ValueError("Invalid learning rate or batch size.")
    if result["max_epochs"] < 1 or result["patience"] < 1:
        raise ValueError("Invalid epoch limit or patience.")
    return result


def config_id(cfg):
    return value_hash(cfg)[:12]


def require_kaggle_gpu():
    if not Path("/kaggle/working").is_dir() or not torch.cuda.is_available():
        raise RuntimeError("Training is allowed only on Kaggle with a GPU enabled.")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def environment():
    return {
        "python": platform.python_version(), "torch": torch.__version__,
        "numpy": np.__version__, "pandas": pd.__version__,
        "sklearn": sklearn.__version__, "joblib": joblib.__version__,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }


def find_input(filename):
    matches = sorted(Path("/kaggle/input").rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Attach exactly one matching output for {filename}; found: {matches}"
        )
    return matches[0]


def validate_train(train):
    if ID not in train or TARGET not in train or train[ID].isna().any():
        raise ValueError("Train must have non-null id and is_fraud.")
    if train[ID].duplicated().any() or set(train[TARGET].dropna().unique()) != {0, 1}:
        raise ValueError("Train IDs must be unique and target must contain both 0 and 1.")
    if train[TARGET].isna().any():
        raise ValueError("Target labels cannot be missing.")


def verify_source(train_path, manifest):
    if file_hash(train_path) != manifest["train_sha256"]:
        raise ValueError("Train data differs from the task 2.6 foundation.")


def holdout_indices(train, split):
    if split[ID].duplicated().any() or set(split[ID]) != set(train[ID]):
        raise ValueError("Split IDs differ from train.")
    labels = split.set_index(ID)["split"].reindex(train[ID]).to_numpy()
    if set(labels) != {"train", "validation"}:
        raise ValueError("Unknown or missing split membership.")
    return np.flatnonzero(labels == "train"), np.flatnonzero(labels == "validation")


def check_probability(probability, n_rows):
    p = np.asarray(probability, dtype=np.float64)
    if p.shape != (n_rows,) or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Invalid prediction length, shape, or probability values.")
    return p


def scores(y, probability):
    p = check_probability(probability, len(y))
    if len(np.unique(y)) != 2:
        raise ValueError("ROC-AUC requires both target classes.")
    return {"roc_auc": float(roc_auc_score(y, p)),
            "average_precision": float(average_precision_score(y, p))}


## 2.6.3 — Preprocessor: fit trên train của từng split/fold

Lưu schema, median, scaler, category mapping và các biến đổi log/chu kỳ. Mã 0 dành cho category chưa thấy. Boolean có thể chuẩn hóa hoặc giữ 0/1. model_spec ghi kích thước thực tế để inference dựng lại đúng mạng.


In [ ]:
%%writefile -a /kaggle/working/task_2_6_foundation/credit_fraud_dl_engine.py
class Preprocessor:
    """Fit exclusively on training rows; save all transforms with each checkpoint."""
    def __init__(self, cfg):
        self.cfg = copy.deepcopy(cfg)

    def _clean(self, raw):
        if list(raw.columns) != self.raw_columns:
            raise ValueError("Feature columns or their order differ from training.")
        frame = raw.copy()
        for col in self.boolean:
            frame[col] = frame[col].astype("float32")
        for col in self.categorical:
            frame[col] = frame[col].astype("string").fillna("__MISSING__").astype(str)
        for col in self.numeric:
            frame[col] = pd.to_numeric(frame[col], errors="coerce").replace(
                [np.inf, -np.inf], np.nan)
        for col in self.cfg["log_columns"]:
            if col not in self.numeric:
                raise ValueError(f"log1p feature is not numeric: {col}")
            if (frame[col].dropna() < 0).any():
                raise ValueError(f"log1p experiment requires non-negative values: {col}")
            frame[col] = np.log1p(frame[col])
        if self.cfg["cyclical"]:
            for col, period in [("time_of_day_hour", 24), ("day_of_week", 7)]:
                if col not in self.numeric:
                    raise ValueError(f"Cyclical feature missing: {col}")
                values = frame[col]
                if ((values.dropna() < 0) | (values.dropna() > period)).any():
                    raise ValueError(f"Unexpected range for cyclical feature: {col}")
                frame[f"{col}_sin"] = np.sin(2 * np.pi * values / period)
                frame[f"{col}_cos"] = np.cos(2 * np.pi * values / period)
        return frame

    def fit(self, raw):
        self.raw_columns = raw.columns.tolist()
        if ID in self.raw_columns or TARGET in self.raw_columns:
            raise ValueError("id and target must be removed before preprocessing.")
        self.boolean = raw.select_dtypes(include=["bool"]).columns.tolist()
        self.categorical = raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
        self.numeric = [c for c in self.raw_columns if c not in self.categorical]
        clean = self._clean(raw)
        self.numeric_out = [c for c in clean.columns if c not in self.categorical]
        self.medians = clean[self.numeric_out].median()
        if self.medians.isna().any():
            raise ValueError("A numerical column is entirely missing in training.")
        self.scaled = [c for c in self.numeric_out
                       if self.cfg["scale_boolean"] or c not in self.boolean]
        self.unscaled = [c for c in self.numeric_out if c not in self.scaled]
        self.scaler = StandardScaler().fit(clean[self.scaled].fillna(self.medians))
        self.mapping = {
            c: {v: i + 1 for i, v in enumerate(sorted(clean[c].unique()))}
            for c in self.categorical
        }
        self.cat_dims = [len(self.mapping[c]) + 1 for c in self.categorical]
        if self.cfg["family"] == "mlp" and self.categorical:
            self.encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                                         dtype=np.float32).fit(clean[self.categorical])
        return self

    def transform(self, raw):
        clean = self._clean(raw)
        numeric = clean[self.numeric_out].fillna(self.medians)
        parts = [self.scaler.transform(numeric[self.scaled]).astype(np.float32)]
        if self.unscaled:
            parts.append(numeric[self.unscaled].to_numpy(dtype=np.float32))
        num = np.concatenate(parts, axis=1)
        cat = np.column_stack([
            clean[c].map(self.mapping[c]).fillna(0).to_numpy(dtype=np.int64)
            for c in self.categorical
        ]) if self.categorical else np.empty((len(clean), 0), dtype=np.int64)
        if self.cfg["family"] == "mlp":
            arr = np.concatenate([num, self.encoder.transform(clean[self.categorical])],
                                 axis=1) if self.categorical else num
            arrays = [arr.astype(np.float32)]
        elif self.cfg["family"] == "embedding":
            arrays = [num, cat]
        else:
            arrays = [np.concatenate([num, cat.astype(np.float32)], axis=1)]
        if not all(np.isfinite(x).all() for x in arrays):
            raise ValueError("Non-finite features after preprocessing.")
        return arrays

    def model_spec(self, arrays):
        if self.cfg["family"] == "mlp":
            return {"input_dim": int(arrays[0].shape[1])}
        if self.cfg["family"] == "embedding":
            dim = self.cfg["embedding_dim"]
            return {"n_numeric": int(arrays[0].shape[1]), "cat_dims": self.cat_dims,
                    "embedding_dims": [
                        int(dim) if dim is not None else min(16, max(2, int(np.ceil(np.sqrt(n)))))
                        for n in self.cat_dims]}
        return {"cat_idxs": list(range(len(self.numeric_out),
                                      len(self.numeric_out) + len(self.categorical))),
                "cat_dims": self.cat_dims}


## 2.6.4 — MLP, embedding và dự đoán theo batch

MLP dùng fully connected, ReLU và dropout; embedding MLP nối vector category với numeric. Dự đoán chia batch để không nạp toàn bộ validation lên VRAM. TabNet chỉ cài khi được chọn, cần Internet hoặc package có sẵn.


In [ ]:
%%writefile -a /kaggle/working/task_2_6_foundation/credit_fraud_dl_engine.py
def mlp_layers(input_dim, cfg):
    layers = []
    for width, dropout in zip(cfg["hidden"], cfg["dropout"]):
        layers.extend([nn.Linear(input_dim, width), nn.ReLU(), nn.Dropout(dropout)])
        input_dim = width
    layers.append(nn.Linear(input_dim, 1))
    return nn.Sequential(*layers)


class EmbeddingMLP(nn.Module):
    def __init__(self, spec, cfg):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(n, d) for n, d in zip(spec["cat_dims"], spec["embedding_dims"])
        ])
        self.network = mlp_layers(spec["n_numeric"] + sum(spec["embedding_dims"]), cfg)

    def forward(self, numeric, categorical):
        embeddings = [layer(categorical[:, i]) for i, layer in enumerate(self.embeddings)]
        return self.network(torch.cat([numeric, *embeddings], dim=1))


def build_mlp(spec, cfg):
    return (EmbeddingMLP(spec, cfg) if cfg["family"] == "embedding"
            else mlp_layers(spec["input_dim"], cfg))


def tensor(array, device="cpu"):
    dtype = torch.long if np.issubdtype(array.dtype, np.integer) else torch.float32
    return torch.as_tensor(array, dtype=dtype, device=device)


def predict_mlp(model, arrays, device, batch_size=8192):
    model.eval()
    parts = []
    with torch.inference_mode():
        for start in range(0, len(arrays[0]), batch_size):
            features = [tensor(a[start:start + batch_size], device) for a in arrays]
            parts.append(torch.sigmoid(model(*features).squeeze(1)).cpu().numpy())
    return check_probability(np.concatenate(parts), len(arrays[0]))


def tabnet_class():
    try:
        from pytorch_tabnet.tab_model import TabNetClassifier
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "pytorch-tabnet==4.1.0"])
        from pytorch_tabnet.tab_model import TabNetClassifier
    return TabNetClassifier


## 2.6.5 — Vòng huấn luyện, scheduler và checkpoint

Mỗi thí nghiệm seed lại RNG, fit preprocessor riêng, train bằng BCE/AdamW hoặc TabNet, chọn checkpoint theo validation AUC. Scheduler giảm learning rate khi AUC chững lại. Mỗi run lưu đủ config, environment, checksum và lịch sử. Chỉ reuse run đã hoàn tất khi signature và mọi artifact đều khớp.


In [ ]:
%%writefile -a /kaggle/working/task_2_6_foundation/credit_fraud_dl_engine.py
def train_run(train, train_idx, valid_idx, cfg, output_dir, source_hash):
    require_kaggle_gpu()
    cfg = config(**cfg)
    train_idx, valid_idx = np.asarray(train_idx), np.asarray(valid_idx)
    if np.intersect1d(train_idx, valid_idx).size:
        raise ValueError("Training and validation rows overlap.")
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    signature = value_hash({
        "engine": file_hash(__file__), "data": source_hash, "config": cfg,
        "training_ids": train.iloc[train_idx][ID].tolist(),
        "validation_ids": train.iloc[valid_idx][ID].tolist(),
    })
    if (output / "metrics.json").is_file():
        existing = json_read(output / "metrics.json")
        if existing["run_signature"] != signature:
            raise ValueError("Output directory belongs to another experiment.")
        if existing.get("reload_verified", False):
            for filename, expected_hash in existing["artifact_sha256"].items():
                if file_hash(output / filename) != expected_hash:
                    raise ValueError(f"Incomplete or changed experiment artifact: {filename}")
            print("Reuse completed run:", output.name)
            return existing
        print("Restart run without completed checkpoint reload verification:", output.name)
    seed_everything(cfg["training_seed"])
    start = time.time()
    X = train.drop(columns=[ID, TARGET])
    y_train = train.iloc[train_idx][TARGET].to_numpy(dtype=np.float32)
    y_valid = train.iloc[valid_idx][TARGET].to_numpy(dtype=np.int64)
    prep = Preprocessor(cfg).fit(X.iloc[train_idx])
    train_arrays, valid_arrays = prep.transform(X.iloc[train_idx]), prep.transform(X.iloc[valid_idx])
    spec = prep.model_spec(train_arrays)
    joblib.dump(prep, output / "preprocessor.joblib")
    positive_weight = (float((y_train == 0).sum() / (y_train == 1).sum())
                       if cfg["positive_weight"] == "balanced" else float(cfg["positive_weight"]))
    if not np.isfinite(positive_weight) or positive_weight <= 0:
        raise ValueError("Invalid positive loss weight.")
    if cfg["family"] == "tabnet":
        Model = tabnet_class()
        model = Model(**{k: v for k, v in cfg["tabnet"].items() if k != "virtual_batch_size"},
                      **spec, optimizer_fn=torch.optim.Adam,
                      optimizer_params={"lr": cfg["learning_rate"], "weight_decay": cfg["weight_decay"]},
                      mask_type="entmax", seed=cfg["training_seed"], verbose=10,
                      device_name="cuda")
        model.fit(train_arrays[0], y_train.astype(np.int64),
                  eval_set=[(valid_arrays[0], y_valid)], eval_name=["valid"],
                  eval_metric=["auc"], max_epochs=cfg["max_epochs"],
                  patience=cfg["patience"], batch_size=cfg["batch_size"],
                  virtual_batch_size=cfg["tabnet"]["virtual_batch_size"],
                  num_workers=0, drop_last=False)
        probability = check_probability(model.predict_proba(valid_arrays[0])[:, 1], len(y_valid))
        model.save_model(str(output / "model"))
        model_file = "model.zip"
        best_epoch = int(model.best_epoch) + 1
        history = pd.DataFrame({k: list(v) for k, v in model.history.history.items()})
    else:
        device = torch.device("cuda:0")
        model = build_mlp(spec, cfg).to(device)
        loader = DataLoader(
            TensorDataset(*[tensor(a) for a in train_arrays],
                          torch.as_tensor(y_train.reshape(-1, 1))),
            batch_size=cfg["batch_size"], shuffle=True, num_workers=0, pin_memory=True,
            generator=torch.Generator().manual_seed(cfg["training_seed"]),
        )
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([positive_weight], dtype=torch.float32, device=device))
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"],
                                      weight_decay=cfg["weight_decay"])
        scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=cfg["scheduler_factor"],
            patience=cfg["scheduler_patience"], min_lr=cfg["min_learning_rate"],
            threshold=cfg["min_delta"], threshold_mode="abs",
        ) if cfg["scheduler"] else None)
        best_auc, stale, best_epoch, best_state = -np.inf, 0, 0, None
        records = []
        for epoch in range(1, cfg["max_epochs"] + 1):
            model.train()
            total_loss, seen = 0.0, 0
            lr_used = optimizer.param_groups[0]["lr"]
            for batch in loader:
                *features, target = [b.to(device, non_blocking=True) for b in batch]
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(*features), target)
                if not torch.isfinite(loss):
                    raise ValueError("Training loss is non-finite.")
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                optimizer.step()
                total_loss += float(loss.detach()) * len(target)
                seen += len(target)
            probability = predict_mlp(model, valid_arrays, device)
            metric = scores(y_valid, probability)
            records.append({"epoch": epoch, "train_loss": total_loss / seen,
                            "learning_rate": lr_used, **metric})
            if metric["roc_auc"] > best_auc + cfg["min_delta"]:
                best_auc, stale, best_epoch = metric["roc_auc"], 0, epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
            if scheduler is not None:
                scheduler.step(metric["roc_auc"])
            print(f"{output.name} epoch={epoch:02d} AUC={metric['roc_auc']:.6f} "
                  f"AP={metric['average_precision']:.6f} lr={lr_used:.2g}")
            if stale >= cfg["patience"]:
                break
        model.load_state_dict(best_state)
        probability = predict_mlp(model, valid_arrays, device)
        model_file = "model.pt"
        torch.save({"state_dict": best_state, "model_spec": spec, "config": cfg},
                   output / model_file)
        history = pd.DataFrame(records)
    history.to_csv(output / "history.csv", index=False)
    predictions = pd.DataFrame({ID: train.iloc[valid_idx][ID].to_numpy(),
                                TARGET: y_valid, "probability": probability})
    predictions.to_csv(output / "validation.csv", index=False)
    filenames = [model_file, "preprocessor.joblib", "history.csv", "validation.csv"]
    result = {
        "config_id": config_id(cfg), "config": cfg, "model_spec": spec,
        "run_signature": signature, "engine_sha256": file_hash(__file__),
        "source_sha256": source_hash, "model_file": model_file,
        "best_epoch": best_epoch, "epochs_run": len(history),
        "positive_weight_used": positive_weight, "environment": environment(),
        "train_rows": len(train_idx), "validation_rows": len(valid_idx),
        "training_seconds": time.time() - start, "reload_verified": False,
        **scores(y_valid, probability),
        "artifact_sha256": {name: file_hash(output / name) for name in filenames},
    }
    json_write(output / "metrics.json", result)
    del model, prep, train_arrays, valid_arrays
    gc.collect()
    torch.cuda.empty_cache()
    reloaded_probability = predict_run(output, X.iloc[valid_idx])
    if not np.allclose(probability, reloaded_probability, atol=1e-6, rtol=1e-5):
        raise ValueError("Reloaded checkpoint/preprocessor changes validation predictions.")
    result["reload_verified"] = True
    json_write(output / "metrics.json", result)
    return result


## 2.6.6 — Inference và kiểm tra submission

Nạp checkpoint cùng preprocessor, kiểm tra cấu hình/checksum, rồi ghép xác suất theo ID của sample. Không đưa ID hoặc target vào mạng; không threshold xác suất trước khi nộp.


In [ ]:
%%writefile -a /kaggle/working/task_2_6_foundation/credit_fraud_dl_engine.py
def predict_run(run_dir, raw_features, device=None):
    run = Path(run_dir)
    metadata = json_read(run / "metrics.json")
    if metadata["engine_sha256"] != file_hash(__file__):
        raise ValueError("Inference engine differs from the training engine.")
    if metadata["environment"]["sklearn"] != sklearn.__version__:
        raise ValueError("Use the same sklearn version as training for joblib preprocessing.")
    for filename in [metadata["model_file"], "preprocessor.joblib"]:
        if file_hash(run / filename) != metadata["artifact_sha256"][filename]:
            raise ValueError(f"Checkpoint/preprocessing checksum mismatch: {filename}")
    prep = joblib.load(run / "preprocessor.joblib")
    if prep.cfg != metadata["config"]:
        raise ValueError("Preprocessor configuration differs from model configuration.")
    arrays = prep.transform(raw_features)
    cfg = metadata["config"]
    if cfg["family"] == "tabnet":
        Model = tabnet_class()
        model = Model(device_name="cuda" if torch.cuda.is_available() else "cpu")
        model.load_model(str(run / metadata["model_file"]))
        probability = check_probability(model.predict_proba(arrays[0])[:, 1], len(raw_features))
    else:
        device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        checkpoint = torch.load(run / metadata["model_file"], map_location="cpu", weights_only=True)
        if checkpoint["config"] != cfg or checkpoint["model_spec"] != metadata["model_spec"]:
            raise ValueError("Checkpoint specification does not match metadata.")
        model = build_mlp(checkpoint["model_spec"], cfg).to(device)
        model.load_state_dict(checkpoint["state_dict"])
        probability = predict_mlp(model, arrays, device)
    del model, arrays, prep
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return probability


def ranked(results):
    return sorted(results, key=lambda r: (r["roc_auc"], r["average_precision"]), reverse=True)


def prediction_frame(path, train):
    frame = pd.read_csv(path)
    if frame[ID].duplicated().any() or set(frame[ID]) != set(train[ID]):
        raise ValueError("OOF IDs do not cover train exactly once.")
    frame = frame.set_index(ID).loc[train[ID]].reset_index()
    if not np.array_equal(frame[TARGET].to_numpy(), train[TARGET].to_numpy()):
        raise ValueError("OOF labels differ from train after ID alignment.")
    check_probability(frame["probability"], len(train))
    return frame


def export_submission(test, sample, probability, destination):
    p = check_probability(probability, len(test))
    if list(sample.columns) != [ID, TARGET] or test[ID].isna().any() or sample[ID].isna().any():
        raise ValueError("Invalid sample schema or missing test/sample IDs.")
    if test[ID].duplicated().any() or sample[ID].duplicated().any() or set(sample[ID]) != set(test[ID]):
        raise ValueError("Test and sample IDs differ or are duplicated.")
    submission = sample[[ID]].copy()
    submission[TARGET] = submission[ID].map(pd.Series(p, index=test[ID]))
    check_probability(submission[TARGET], len(sample))
    submission.to_csv(destination, index=False)
    return submission


## 2.6.7 — Audit, giữ split cũ và tạo CV folds

Đọc split theo ID từ 2.1, không chia ngẫu nhiên lại. Audit ghi tỷ lệ lớp, missing/infinite, skewness và hàng trùng feature.
Nếu có feature vector trùng, CV dùng StratifiedGroupKFold để cùng nhóm không nằm cả train và validation; không xem ID là thời gian.
Nếu không có trùng, dùng StratifiedKFold 5-fold, shuffle seed 9. Holdout cũ vẫn giữ để so sánh lịch sử và báo số nhóm trùng xuyên split nếu có.


In [ ]:
import importlib.util
import sys
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

module_path = OUTPUT_DIR / "credit_fraud_dl_engine.py"
spec = importlib.util.spec_from_file_location("credit_fraud_dl_engine", module_path)
dl = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dl
spec.loader.exec_module(dl)

DATA_DIR = None  # Auto-detect inputs; set a directory here only to disambiguate.

def resolve_data_dir(data_dir=None, expected_sha256=None,
                     required_files=("train.csv",), input_root="/kaggle/input"):
    """Find attached data without assuming Kaggle's mount layout."""
    from pathlib import Path
    import hashlib

    root = Path(input_root)
    if data_dir is not None:
        candidates = [Path(data_dir)]
        missing = [name for name in required_files if not (candidates[0] / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"DATA_DIR={data_dir} is missing {missing}. "
                "Attach competition data or set DATA_DIR=None for automatic discovery."
            )
    else:
        candidates = sorted({
            path.parent for path in root.rglob("train.csv")
            if path.is_file() and all((path.parent / name).is_file() for name in required_files)
        })
        if not candidates:
            mounted = sorted(path.name for path in root.iterdir()) if root.is_dir() else []
            raise FileNotFoundError(
                f"No attached directory contains {list(required_files)} under {root}. "
                "Use Add Input to attach competition playground-series-s6e11. "
                f"Mounted entries: {mounted}"
            )

    if expected_sha256 is not None:
        matching = []
        for directory in candidates:
            digest = hashlib.sha256()
            with (directory / "train.csv").open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() == expected_sha256:
                matching.append(directory)
        if not matching:
            raise ValueError(
                "Attached train.csv does not match the task 2.6 checksum. "
                f"Check the competition data/version. Candidates: {candidates}"
            )
        candidates = matching
    elif len(candidates) > 1:
        bundles = [directory for directory in candidates
                   if all((directory / name).is_file()
                          for name in ("train.csv", "test.csv", "sample_submission.csv"))]
        if bundles:
            candidates = bundles

    if len(candidates) != 1:
        raise ValueError(
            f"Multiple valid data directories found: {candidates}. "
            "Set DATA_DIR to the exact directory you want to use."
        )
    return candidates[0]

DATA_DIR = resolve_data_dir(DATA_DIR)
print("Resolved DATA_DIR:", DATA_DIR)
train_path = DATA_DIR / "train.csv"
train = pd.read_csv(train_path)
dl.validate_train(train)
old_split_path = dl.find_input("split_ids_seed9.csv")
split = pd.read_csv(old_split_path)
train_idx, valid_idx = dl.holdout_indices(train, split)
split.to_csv(OUTPUT_DIR / "holdout_split_seed9.csv", index=False)

features = train.drop(columns=[dl.ID, dl.TARGET])
group_hash = pd.util.hash_pandas_object(features, index=False).astype(str)
has_duplicates = bool(group_hash.duplicated().any())
fold_ids = np.full(len(train), -1, dtype=np.int64)
if has_duplicates:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=9)
    partitions = splitter.split(features, train[dl.TARGET], groups=group_hash)
    cv_method = "StratifiedGroupKFold (identical feature vectors grouped)"
else:
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=9)
    partitions = splitter.split(features, train[dl.TARGET])
    cv_method = "StratifiedKFold"
for fold, (fit_idx, val_idx) in enumerate(partitions):
    if train.iloc[val_idx][dl.TARGET].nunique() != 2 or train.iloc[fit_idx][dl.TARGET].nunique() != 2:
        raise ValueError("A CV partition does not contain both classes.")
    if set(group_hash.iloc[fit_idx]) & set(group_hash.iloc[val_idx]):
        raise ValueError("Duplicate features cross a CV fold.")
    fold_ids[val_idx] = fold
if (fold_ids < 0).any():
    raise ValueError("CV membership is incomplete.")
pd.DataFrame({dl.ID: train[dl.ID], "fold": fold_ids, "feature_group": group_hash}).to_csv(
    OUTPUT_DIR / "cv_folds_seed9.csv", index=False
)
numeric = features.select_dtypes(include=["number"])
infinite = numeric.isin([np.inf, -np.inf]).sum()
audit = {
    "rows": len(train), "features": features.columns.tolist(),
    "positive_count": int(train[dl.TARGET].sum()),
    "positive_rate": float(train[dl.TARGET].mean()),
    "missing_by_column": {c: int(n) for c, n in train.isna().sum().items()},
    "infinite_by_numeric_column": {c: int(n) for c, n in infinite.items()},
    "duplicate_feature_rows_beyond_first": int(group_hash.duplicated().sum()),
    "duplicate_groups_across_old_holdout": len(
        set(group_hash.iloc[train_idx]) & set(group_hash.iloc[valid_idx])),
    "cv_method": cv_method,
    "cv_folds": 5,
    "limitations": "No customer/card grouping or chronological order inferred from id.",
}
numeric.describe().to_csv(OUTPUT_DIR / "numeric_summary.csv")
numeric.skew().rename("skewness").to_csv(OUTPUT_DIR / "numeric_skewness.csv")
dl.json_write(OUTPUT_DIR / "audit.json", audit)

manifest = {
    "engine_version": dl.ENGINE_VERSION, "engine_sha256": dl.file_hash(module_path),
    "train_sha256": dl.file_hash(train_path), "split_seed": 9, "cv_method": cv_method,
    "membership_sha256": {name: dl.file_hash(OUTPUT_DIR / name) for name in
                         ["holdout_split_seed9.csv", "cv_folds_seed9.csv"]},
    "environment": dl.environment(),
}
manifest["foundation_id"] = dl.value_hash(manifest)
dl.json_write(OUTPUT_DIR / "foundation_manifest.json", manifest)
display(pd.DataFrame({"fold": fold_ids, "target": train[dl.TARGET]}).groupby("fold").agg(
    rows=("target", "size"), fraud_rate=("target", "mean")))
print(audit)


## 2.6.8 — Baseline tham chiếu mới

Chạy cấu hình gốc 12 epoch/patience 3, không scheduler; lưu validation theo ID và toàn bộ artifact.
Điểm baseline cũ 0.944622 là kết quả lịch sử; baseline chạy lại có thể lệch do phiên bản môi trường. Các task tối ưu so sánh với lượt chạy mới này.
Fold membership đã được cố định trước tuning; task 2.9 sẽ train baseline và ứng viên trên cùng folds.


In [ ]:
baseline = dl.train_run(
    train, train_idx, valid_idx, dl.config(),
    OUTPUT_DIR / "baseline_holdout", manifest["train_sha256"],
)
dl.json_write(OUTPUT_DIR / "baseline_reference.json", {
    "foundation_id": manifest["foundation_id"],
    "historical_holdout_roc_auc": 0.9446222144255438,
    "rerun": baseline,
})
print("Baseline re-run:", baseline["roc_auc"], baseline["average_precision"])
print("Save Version / Run All hoàn tất rồi gắn output này vào notebook 2.7.")
